# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [22]:
aviation_df = pd.read_csv('data/AviationData.csv', index_col=0, encoding='latin1')
state_code_df = pd.read_csv('data/USState_Codes.csv', index_col=0)

/var/folders/xc/f61f2m_96nl7n_93qmnbncgm0000gn/T/ipykernel_74354/1663000573.py:1: DtypeWarning: Columns (6,7,28) have mixed types. Specify dtype option on import or set low_memory=False.
  aviation_df = pd.read_csv('data/AviationData.csv', index_col=0, encoding='latin1')


In [23]:
#aviation_df.shape
#state_code_df.shape

#aviation_df.info()
#state_code_df.info()

aviation_df.head()
#state_code_df.head()

#aviation_df["Amateur.Built"]
aviation_df["Aircraft.Category"]

Event.Id
20001218X45444         NaN
20001218X45447         NaN
20061025X01555         NaN
20001218X45448         NaN
20041105X01764         NaN
                    ...   
20221227106491         NaN
20221227106494         NaN
20221227106497    Airplane
20221227106498         NaN
20221230106513         NaN
Name: Aircraft.Category, Length: 88889, dtype: object

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [24]:


filtered_aviation_df = aviation_df[(aviation_df["Amateur.Built"] == 'No') & (aviation_df["Aircraft.Category"] == "Airplane") & (pd.to_datetime(aviation_df["Event.Date"]) >= pd.Timestamp.now() - pd.DateOffset(years=40))].copy()
filtered_aviation_df.head()

,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,Injury.Severity,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
Event.Id,,,,,,,,,,,,,,,,,,,,,
20001213X33489,Incident,BFO86IA031B,1986-05-30,"CALVERTON, NY",United States,NaN,NaN,NaN,NaN,Incident,...,Public Aircraft,NaN,NaN,NaN,NaN,69.0,VMC,Descent,Probable Cause,09-10-2018
20001213X33489,Incident,BFO86IA031A,1986-05-30,"CALVERTON, NY",United States,NaN,NaN,NaN,NaN,Incident,...,Unknown,NaN,NaN,NaN,NaN,69.0,VMC,Descent,Probable Cause,09-10-2018
20001213X30060,Accident,DCA87MA018B,1987-01-15,"KEARNS, UT",United States,NaN,NaN,NaN,NaN,Fatal(10),...,Instructional,NaN,10.0,NaN,NaN,NaN,VMC,Maneuvering,Probable Cause,10-07-2019
20001213X30060,Accident,DCA87MA018A,1987-01-15,"KEARNS, UT",United States,NaN,NaN,NaN,NaN,Fatal(10),...,Unknown,Sky West Airlines Inc. (dba: Sky West Airlines...,10.0,NaN,NaN,NaN,VMC,Maneuvering,Probable Cause,10-07-2019
20001213X30244,Accident,FTW87RA066,1987-02-14,"DURANGO, Mexico",Mexico,NaN,NaN,NaN,NaN,Fatal(1),...,Unknown,NaN,1.0,NaN,3.0,131.0,UNK,NaN,Foreign,07-02-1995


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [25]:
num_of_passengers = [
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured"
]

filtered_aviation_df[num_of_passengers] = filtered_aviation_df[num_of_passengers].fillna(0.0)

# Grab and estimate of total # of passengers based on present data and store in new "Total.Onboard" column
filtered_aviation_df["Total.Onboard"] = (
    filtered_aviation_df["Total.Fatal.Injuries"] +
    filtered_aviation_df["Total.Serious.Injuries"] +
    filtered_aviation_df["Total.Minor.Injuries"] +
    filtered_aviation_df["Total.Uninjured"]
)

# Create a count of Fatal or Serious injuries and store in new "Fatal.Serious.Count" column
filtered_aviation_df["Fatal.Serious.Count"] = (
    filtered_aviation_df["Total.Fatal.Injuries"] +
    filtered_aviation_df["Total.Serious.Injuries"]
)

# Compute the rate of Fatal and Serious injuries by dividing it with the total # of passengers on board
filtered_aviation_df["Fatal.Serious.Rate"] = (
    filtered_aviation_df["Fatal.Serious.Count"] /
    filtered_aviation_df["Total.Onboard"]
)

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [26]:
filtered_aviation_df["Aircraft.damage"] = filtered_aviation_df["Aircraft.damage"].fillna("Unknown")

filtered_aviation_df["Aircraft.Damage"] =  (filtered_aviation_df["Aircraft.damage"] == "Destroyed").astype(int)

filtered_aviation_df["Aircraft.Damage"].head()

Event.Id
20001213X33489    0
20001213X33489    0
20001213X30060    1
20001213X30060    1
20001213X30244    0
Name: Aircraft.Damage, dtype: int64

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [ ]:
#filtered_aviation_df["Make"]

# I don't know if null values exist here but just in case I replace them with unknown
filtered_aviation_df["Make"] = filtered_aviation_df["Make"].fillna("Unknown")

# Make all Make's lowercase as some of them are all uppercase and others all lowercase
filtered_aviation_df["Make"] = filtered_aviation_df["Make"].str.lower()

# Strip leading and closing whitespace
filtered_aviation_df["Make"] = filtered_aviation_df["Make"].str.strip()

# A couple similar values I found in the data which replace with a normalized value
filtered_aviation_df["Make"] = filtered_aviation_df["Make"].replace({
    "cirrus design corp": "cirrus",
    "cirrus design corp.": "cirrus",
    "aviat aircraft inc": "aviat",
    "rockwell international": "rockwell",
    "mooney aircraft corp.": "mooney",
    "grumman acft eng cor-schweizer": "grumman",
    "grumman american avn. corp.": "grumman",
    "grumman american": "grumman"
    })

make_counts = filtered_aviation_df["Make"].value_counts()

# Keep only makes with > 50 occurrences
valid_makes = make_counts[make_counts >= 50].index

# Apply valid makes to filtered aviation df to remove invalid makes
filtered_aviation_df = filtered_aviation_df[
    filtered_aviation_df["Make"].isin(valid_makes)
]

filtered_aviation_df.groupby("Make").size()

# Quick check to see any possible data inconsistency. Change head with tail
#filtered_aviation_df["Make"].value_counts().head(50)

Make
aero commander                  90
aeronca                        200
air tractor                    206
air tractor inc                219
airbus                         243
american champion aircraft      52
aviat                          146
ayres                           55
beech                         1429
bellanca                       219
boeing                        1262
bombardier inc                  65
cessna                        7136
champion                       158
cirrus                         385
de havilland                    72
dehavilland                     95
diamond aircraft ind inc        74
embraer                        153
ercoupe                         52
grumman                        277
luscombe                       141
maule                          215
mcdonnell douglas              108
mooney                         397
north american                 106
piper                         3985
raytheon aircraft company       62
rockwell       

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [38]:
filtered_aviation_df["Model"].dropna()

filtered_aviation_df["Model"] = (
    filtered_aviation_df["Model"]
    .fillna("unknown")
    .str.lower()
    .str.strip()
)

filtered_aviation_df["Make.Model"] = (
    filtered_aviation_df["Make"] + "-" + filtered_aviation_df["Model"]
)

filtered_aviation_df["Make.Model"]

Event.Id
20001213X33489                       grumman-f-14a
20001213X30060                        mooney-m-20c
20001213X30244                     boeing-707-323b
20001213X30820                         piper-pa 16
20001213X25245                         piper-pa-38
                                ...               
20221212106444                          cessna-172
20221213106455                          piper-pa42
20221215106463                         cirrus-sr22
20221219106470                        cessna-r172k
20221227106497    american champion aircraft-8gcbc
Name: Make.Model, Length: 17999, dtype: object

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [ ]:
#filtered_aviation_df["Weather.Condition"].value_counts(dropna=False)

filtered_aviation_df["Weather.Condition"] = filtered_aviation_df["Weather.Condition"].str.upper()

Broad.phase.of.flight
NaN            15528
Landing         1120
Takeoff          428
Cruise           242
Approach         210
Maneuvering      132
Taxi              97
Go-around         83
Descent           61
Climb             52
Standing          33
Unknown           11
Other              2
Name: count, dtype: int64

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [49]:
filtered_aviation_df["Broad.phase.of.flight"].value_counts(dropna=False)

filtered_aviation_df = filtered_aviation_df.drop(columns=["Broad.phase.of.flight"])

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [50]:
filtered_aviation_df.to_csv("cleaned_aviation_data.csv", index=False)